# Misión 01 · ¿Dónde blindar los aviones?

**The Agentic D. Scientist** · Nivel 1 · Sesgo de supervivencia

No necesitas saber programar. En cada celda gris presiona **▶** (o `Shift + Enter`) y lee lo que aparece.

> Los aviones de esta misión son **simulados**: sirven para ver el mecanismo, no reproducen cifras históricas. La historia real está al final.

## 1. Trae los datos

Cada noche salen 1,000 aviones. Solo vemos los que **regresan**. Esta celda descarga sus impactos por parte del avión.

In [ ]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/jcval94/mvp-ds-course/main/casos/data/wald-bombers/"
LOCAL = Path("casos/data/wald-bombers")  # si abres el cuaderno dentro del repositorio

def fuente(nombre):
    return str(LOCAL / nombre) if (LOCAL / nombre).exists() else BASE + nombre

def leer_json(nombre):
    ruta = fuente(nombre)
    if ruta.startswith("http"):
        with urllib.request.urlopen(ruta) as respuesta:
            return json.load(respuesta)
    return json.loads(Path(ruta).read_text(encoding="utf-8"))

parametros = leer_json("parametros.json")
regresaron = pd.read_csv(fuente("regresaron.csv"))
partes = {s["id"]: s["nombre"] for s in parametros["secciones"]}
area = {s["id"]: s["area"] for s in parametros["secciones"]}

print(f"Aviones que regresaron: {len(regresaron)} de {parametros['aviones_por_mision']}")
regresaron.head()

## 2. ¿Dónde tienen más agujeros?

Contamos los impactos de cada parte. Como las alas son mucho más grandes que la cabina, también calculamos los impactos **por cada 1% de la superficie**.

In [ ]:
evidencia = pd.DataFrame({
    "parte": [partes[p] for p in partes],
    "impactos": [int(regresaron[p].sum()) for p in partes],
    "por cada 1% de superficie": [round(regresaron[p].sum() / (area[p] * 100), 1) for p in partes],
})
evidencia

In [ ]:
AMBAR, ROJO, TEXTO = "#fdaa3e", "#d74745", "#0e0a07"  # colores del curso

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(evidencia["parte"], evidencia["por cada 1% de superficie"], color=ROJO)
ax.invert_yaxis()
ax.set_xlabel("Impactos por cada 1% de superficie")
ax.set_title("Aviones que REGRESARON", loc="left", color=TEXTO)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Tu decisión

Tienes **2 placas de blindaje**. Elige dónde ponerlas en el menú de la derecha y luego presiona ▶. La celda envía 1,000 aviones con tu blindaje.

In [ ]:
blindaje = "Fuselaje y alas"  # @param ["Fuselaje y alas", "Alas y cola", "Motores y cabina", "Motores y alas", "Cabina y cola"]

import math
import random

def simular(parametros, blindadas=(), semilla=None):
    # Mismo modelo que casos/data/wald-bombers/generar.py y que la misión web.
    rng = random.Random(parametros["semilla"] if semilla is None else semilla)
    secciones = parametros["secciones"]
    pesos = [s["area"] for s in secciones]
    reduccion = parametros["blindaje"]["reduccion_letalidad"]
    regresan = 0
    for _ in range(parametros["aviones_por_mision"]):
        limite, impactos, producto = math.exp(-parametros["impactos_promedio"]), 0, rng.random()
        while producto > limite:
            impactos += 1
            producto *= rng.random()
        derribado = False
        for _ in range(impactos):
            seccion = rng.choices(secciones, weights=pesos)[0]
            letalidad = seccion["letalidad"] * ((1 - reduccion) if seccion["id"] in blindadas else 1)
            if rng.random() < letalidad:
                derribado = True
        regresan += not derribado
    return regresan

nombres = {v: k for k, v in partes.items()}
elegidas = [nombres[n.strip().capitalize()] for n in blindaje.split(" y ")]
sin_blindaje = simular(parametros)
con_blindaje = simular(parametros, set(elegidas))
print(f"Sin blindaje regresan:  {sin_blindaje}")
print(f"Con {blindaje.lower()}: {con_blindaje}  ({con_blindaje - sin_blindaje:+d})")

¿Cambió mucho? Si blindaste donde había más agujeros, probablemente **no**. Antes de seguir, escribe en una línea por qué crees que pasó.

## 4. Lo que no se veía

En la guerra nadie podía revisar los aviones derribados. Aquí sí: `todos.csv` incluye a los que **no regresaron**.

In [ ]:
todos = pd.read_csv(fuente("todos.csv"))
derribados = todos[todos["regreso"] == "no"]

def por_avion(tabla):
    return [tabla[p].sum() / (area[p] * 10) / len(tabla) for p in partes]

comparacion = pd.DataFrame({
    "parte": [partes[p] for p in partes],
    "regresaron": [round(x, 2) for x in por_avion(regresaron)],
    "derribados": [round(x, 2) for x in por_avion(derribados)],
})
print("Impactos por avión por cada 10% de superficie")
comparacion

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
y = range(len(comparacion))
ax.barh([i - 0.2 for i in y], comparacion["regresaron"], height=0.4, color=ROJO, label="Regresaron")
ax.barh([i + 0.2 for i in y], comparacion["derribados"], height=0.4, color=TEXTO, label="Derribados")
ax.set_yticks(list(y), comparacion["parte"])
ax.invert_yaxis()
ax.set_xlabel("Impactos por avión por cada 10% de superficie")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Los motores y la cabina casi no tienen agujeros en los aviones que regresan **no porque reciban menos disparos**, sino porque los aviones que recibieron ahí **no volvieron**.

Eso es el **sesgo de supervivencia**: tus datos solo incluyen a quienes pasaron un filtro, y el filtro esconde justo lo que buscas.

Regresa a la celda de la sección 3, elige **Motores y cabina** y vuelve a presionar ▶.

## 5. Llévalo a otro caso

Una app de ejercicio encuesta solo a quienes la siguen usando después de 6 meses: 9 de cada 10 dicen que les encanta. ¿Qué **no** puedes concluir, y qué datos te faltan?

## La historia real (sin mitos)

- Abraham Wald trabajó en el Statistical Research Group de la Universidad de Columbia durante la Segunda Guerra Mundial.
- Hacia 1943 escribió memorandos para estimar la vulnerabilidad de un avión a partir de los daños de los que regresaban. Nunca se publicaron externamente; el Center for Naval Analyses los reimprimió en 1980.
- No hay constancia de que dibujara el famoso avión con puntos rojos ni de que ordenara blindar una zona concreta.

Fuentes: [AMS Feature Column](https://www.ams.org/samplings/feature-column/fc-2016-06) · [Center for Naval Analyses](https://cna.org/reports/1980/a-method-of-estimating-plane-vulnerability) · [reimpresión CRC 432](https://apps.dtic.mil/sti/pdfs/ADA091073.pdf)